# TeachAI — ноутбук экспериментов (Этап №5)

**Роль в сдаче:** второй ноутбук этапа №5 — сравнение и выбор модели LLM  
для генерации учебных материалов TeachAI (вместо классического перебора архитектур/эпох).

**Почему не train/val accuracy:** проект — inference на готовой GPT через API;  
обучение весов не выполняется (см. обоснования в корне репозитория).

**Аналог «лучшая модель по точность/параметры»:**  
лучшая конфигурация по **качеству судей (0–10) / цене (₽) / latency**.

| Артефакт | Путь |
|----------|------|
| Модуль бенчмарка | `model_benchmark/` |
| Конфиг участников/судей | `model_benchmark/config.json` |
| Рубрика | `model_benchmark/rubric.json` |
| Сценарии (5 уроков Python) | `model_benchmark/scenarios/lessons.json` |
| Полный прогон (13.09.2026) | `model_benchmark/results/lesson_benchmark_20260913_123645.json` |
| Подробный отчёт | `ОТЧЁТ - Бенчмарк моделей LLM для TeachAI.md` |

По умолчанию ноутбук **читает сохранённый JSON** (воспроизводимо без API).  
Живой повторный прогон — опциональная ячейка в конце (нужен `GPTUNNEL_API_KEY`).

## 1. Функции эксперимента

Чистые функции: принимают пути/данные аргументами, пути имеют значения по умолчанию.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Optional

import matplotlib.pyplot as plt

ROOT = Path.cwd()
DEFAULT_RESULTS = (
    ROOT
    / "model_benchmark"
    / "results"
    / "lesson_benchmark_20260913_123645.json"
)
SMOKE_RESULTS = (
    ROOT
    / "model_benchmark"
    / "results"
    / "lesson_benchmark_20260913_120448.json"
)


def load_benchmark_report(path: Path = DEFAULT_RESULTS) -> dict[str, Any]:
    """Загружает JSON-отчёт бенчмарка с диска."""
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Нет файла результатов: {path}")
    with path.open(encoding="utf-8") as f:
        return json.load(f)


def summarize_leaderboard(report: dict[str, Any]) -> list[dict[str, Any]]:
    """Возвращает leaderboard, отсортированный по avg_total (убыв.)."""
    rows = list(report.get("leaderboard") or [])
    return sorted(rows, key=lambda r: -(r.get("avg_total") or 0))


def criteria_means(report: dict[str, Any]) -> dict[str, dict[str, float]]:
    """Средние баллы по критериям рубрики: participant -> criterion -> mean."""
    criteria = ["topic_fit", "completeness", "density", "pedagogy", "task_fit", "total"]
    buckets: dict[str, dict[str, list[float]]] = {}

    for lesson in report.get("lessons") or []:
        # В JSON полного прогона блок судей называется "judges"
        for judge in lesson.get("judges") or []:
            by_participant = judge.get("by_participant") or {}
            for pid, sc in by_participant.items():
                if not isinstance(sc, dict):
                    continue
                buckets.setdefault(pid, {c: [] for c in criteria})
                for c in criteria:
                    if sc.get(c) is not None:
                        buckets[pid][c].append(float(sc[c]))

    out: dict[str, dict[str, float]] = {}
    for pid, by_c in buckets.items():
        out[pid] = {
            c: (sum(vals) / len(vals) if vals else float("nan"))
            for c, vals in by_c.items()
        }
    return out


def format_gate_summary(report: dict[str, Any]) -> list[dict[str, Any]]:
    """Сводка format gate по всем генерациям."""
    rows = []
    for lesson in report.get("lessons") or []:
        lid = (lesson.get("lesson") or {}).get("id")
        for pid, gen in (lesson.get("generations") or {}).items():
            gate = gen.get("format_gate") or {}
            rows.append(
                {
                    "lesson": lid,
                    "participant": pid,
                    "passed": gate.get("passed"),
                    "chars": gate.get("char_count"),
                    "latency_sec": gen.get("latency_sec"),
                }
            )
    return rows


def plot_quality_and_cost(leaderboard: list[dict[str, Any]]) -> None:
    """Два графика: средний балл судей и средняя цена генерации."""
    labels = [r.get("label") or r.get("participant_id") for r in leaderboard]
    scores = [r.get("avg_total") or 0 for r in leaderboard]
    costs = [r.get("avg_gen_cost_rub") or 0 for r in leaderboard]

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    axes[0].bar(labels, scores, color="#4C78A8")
    axes[0].set_ylabel("Средний total (0..10)")
    axes[0].set_title("Качество (LLM-as-judge)")
    axes[0].set_ylim(0, 10)
    axes[0].tick_params(axis="x", rotation=25)

    axes[1].bar(labels, costs, color="#F58518")
    axes[1].set_ylabel("Средняя цена урока, ₽")
    axes[1].set_title("Стоимость генерации (участник)")
    axes[1].tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.show()


def plot_score_per_ruble(leaderboard: list[dict[str, Any]]) -> None:
    """График отдачи: балл / рубль (выше = выгоднее)."""
    ordered = sorted(leaderboard, key=lambda r: -(r.get("score_per_ruble") or 0))
    labels = [r.get("label") or r.get("participant_id") for r in ordered]
    values = [r.get("score_per_ruble") or 0 for r in ordered]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(labels, values, color="#54A24B")
    ax.set_ylabel("avg_total / avg_gen_cost_rub")
    ax.set_title("Экономическая эффективность (балл на рубль)")
    ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.show()


def plot_latency(leaderboard: list[dict[str, Any]]) -> None:
    """График средней задержки генерации."""
    labels = [r.get("label") or r.get("participant_id") for r in leaderboard]
    vals = [r.get("avg_latency_sec") or 0 for r in leaderboard]

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(labels, vals, color="#E45756")
    ax.set_ylabel("Секунды")
    ax.set_title("Средняя latency генерации урока")
    ax.tick_params(axis="x", rotation=25)
    plt.tight_layout()
    plt.show()


def print_table(leaderboard: list[dict[str, Any]]) -> None:
    """Печатает сводную таблицу результатов."""
    print(f"{'#':<3} {'Модель':<42} {'Балл':>6} {'₽/урок':>8} {'Балл/₽':>8} {'сек':>7}")
    print("-" * 80)
    for i, r in enumerate(leaderboard, 1):
        name = (r.get("label") or r.get("participant_id") or "")[:42]
        print(
            f"{i:<3} {name:<42} "
            f"{(r.get('avg_total') or 0):6.2f} "
            f"{(r.get('avg_gen_cost_rub') or 0):8.2f} "
            f"{(r.get('score_per_ruble') or 0):8.2f} "
            f"{(r.get('avg_latency_sec') or 0):7.1f}"
        )


def print_verdict(leaderboard: list[dict[str, Any]]) -> None:
    """Фиксированный вердикт этапа (по полному прогону 13.09.2026)."""
    by_id = {r.get("participant_id"): r for r in leaderboard}
    mini = by_id.get("gpt-4o-mini", {})
    luna = by_id.get("gpt-5.6-luna", {})

    print("=== ВЕРДИКТ ЭТАПА №5 (выбор модели) ===")
    print("Критерии (зафиксированы до прогона):")
    print("  1) экономическая эффективность (балл/₽)")
    print("  2) приемлемое абсолютное качество")
    print("  3) низкая цена массовой генерации и низкая latency")
    print("  4) прохождение format gate")
    print()
    print(
        f"Лучшая по абсолютному качеству: gpt-5.6-luna "
        f"({luna.get('avg_total')}/10, ~{luna.get('avg_gen_cost_rub')} ₽/урок)"
    )
    print(
        f"Лучшая по баллу/₽ и скорости: gpt-4o-mini "
        f"({mini.get('avg_total')}/10, ~{mini.get('avg_gen_cost_rub')} ₽/урок, "
        f"score_per_ruble={mini.get('score_per_ruble')})"
    )
    print()
    print("РЕШЕНИЕ ДЛЯ УЧЕБНОГО TeachAI:")
    print("  Базовую модель gpt-4o-mini ОСТАВЛЯЕМ (default в проекте).")
    print("  Обоснование: достаточное качество + лучшая отдача на рубль + минимальная latency.")
    print("  Рекомендуемый апгрейд при бюджете: gpt-5.6-luna.")
    print("  Не выбираем для массовых уроков: terra / claude-5-sonnet / gemini-3.5-flash (дорого относительно прироста).")


print("Функции загружены. ROOT =", ROOT)
print("DEFAULT_RESULTS =", DEFAULT_RESULTS)

## 2. Загрузка сохранённого полного прогона

Эксперимент уже выполнен 13.09.2026: 5 уроков × 5 участников × 3 судьи.  
Sample данных лежит рядом: `model_benchmark/results/`.

In [ ]:
report = load_benchmark_report()
leaderboard = summarize_leaderboard(report)

print("created_at:", report.get("created_at"))
print("course:", report.get("course"))
print("lessons:", len(report.get("lessons") or []))
print("participants:", [p.get("model") for p in report.get("participants") or []])
print("judges:", [j.get("model") for j in report.get("judges") or []])
print("cost_summary:", json.dumps(report.get("cost_summary"), ensure_ascii=False, indent=2))
print()
print_table(leaderboard)

## 3. Эксперимент A — format gate (автопроверка формы)

Аналог «модель вообще обучается / выдаёт валидный выход».  
Пороги: ≥800 символов, markdown-заголовки, блок ```python.

In [ ]:
gate_rows = format_gate_summary(report)
n_ok = sum(1 for r in gate_rows if r.get("passed"))
print(f"Format gate: {n_ok}/{len(gate_rows)} генераций прошли")
for row in gate_rows:
    print(row)

## 4. Эксперимент B — качество и цена (полный прогон)

Сравнение пяти моделей на одних и тех же промптах урока TeachAI.

In [ ]:
plot_quality_and_cost(leaderboard)
plot_score_per_ruble(leaderboard)
plot_latency(leaderboard)

## 5. Эксперимент C — разрез по критериям рубрики

Рубрика (0–2): topic_fit, completeness, density, pedagogy, task_fit; total = сумма (0–10).

In [ ]:
means = criteria_means(report)
crit_names = ["topic_fit", "completeness", "density", "pedagogy", "task_fit", "total"]

print(f"{'model':<22} " + " ".join(f"{c:>12}" for c in crit_names))
print("-" * 100)
for pid in sorted(means.keys()):
    vals = means[pid]
    print(
        f"{pid:<22} "
        + " ".join(f"{vals.get(c, float('nan')):12.2f}" for c in crit_names)
    )

# Столбчатая диаграмма pedagogy vs task_fit
pids = list(means.keys())
x = range(len(pids))
fig, ax = plt.subplots(figsize=(11, 4))
w = 0.35
ax.bar([i - w / 2 for i in x], [means[p]["pedagogy"] for p in pids], width=w, label="pedagogy")
ax.bar([i + w / 2 for i in x], [means[p]["task_fit"] for p in pids], width=w, label="task_fit")
ax.set_xticks(list(x))
ax.set_xticklabels(pids, rotation=25)
ax.set_ylim(0, 2.2)
ax.set_ylabel("Средний балл критерия (0..2)")
ax.set_title("Педагогичность vs соответствие заданию")
ax.legend()
plt.tight_layout()
plt.show()

## 6. Эксперимент D — дымовой vs полный прогон

На 1 уроке лидировал Claude; на 5 уроках — Luna.  
Вывод: одного сценария недостаточно для выбора модели.

In [ ]:
if SMOKE_RESULTS.exists():
    smoke = load_benchmark_report(SMOKE_RESULTS)
    smoke_lb = summarize_leaderboard(smoke)
    print("Дымовой прогон (1 урок):")
    print_table(smoke_lb)
    print()
    print("Полный прогон (5 уроков):")
    print_table(leaderboard)
else:
    print("Дымовой JSON не найден:", SMOKE_RESULTS)

## 7. Итоговые выводы (обязательный блок этапа №5)

### Сравнение «архитектур» (моделей API)

| Модель | Балл | ₽/урок | Балл/₽ | Latency | Роль |
|--------|-----:|-------:|-------:|--------:|------|
| gpt-5.6-luna | **8.40** | 0.88 | 9.57 | ~84 с | лучшее качество при умеренной цене |
| gemini-3.5-flash | 8.33 | 6.53 | 1.28 | ~21 с | быстро, но дорого относительно Luna |
| gpt-5.6-terra | 8.27 | 8.78 | 0.94 | ~82 с | почти как Luna, ~10× дороже |
| **gpt-4o-mini** | 7.73 | **0.53** | **14.72** | **~12 с** | **выбранный default TeachAI** |
| claude-5-sonnet | 7.67 | 7.57 | 1.01 | ~37 с | невыгоден для массовых уроков |

### Удалось ли добиться желаемой «точности»?

Формальной ML-точности (accuracy на hold-out) в проекте нет.  
Целевой ориентир этапа: **рабочее качество генерации уроков** при учебной экономике.

- Format gate: **25/25** пройдено.
- Средний балл baseline `gpt-4o-mini`: **7.73/10** — приемлемо, не провал.
- Лидер качества Luna: **8.40/10** при цене всё ещё ~1 ₽/урок.

### Выбранная конфигурация для финальной системы

**`gpt-4o-mini`** — базовая модель генерации уроков в TeachAI.  
Решение экономическое для учебного проекта; апгрейд-кандидат — `gpt-5.6-luna`.

### Ограничения (честно)

- Оценён тип генерации «урок», не весь TI.
- Для судей длинные тексты обрезались до ~3500 символов (см. отчёт бенчмарка).
- LLM-as-judge ≠ идеальный методист; зато воспроизводимо.
- Цены — тарифы GPTunnel на дату прогона.

In [ ]:
print_verdict(leaderboard)

## 8. (Опционально) Живой повторный прогон

Не обязателен для проверки этапа: результаты уже в `results/`.  
Для повтора нужны `GPTUNNEL_API_KEY` в `.env` и расходы на API.

Дым: `lesson_ids=["lesson-variables"]`. Полный: `lesson_ids=None`.

In [ ]:
RUN_LIVE = False  # поставить True только для платного повторного прогона

if RUN_LIVE:
    import os
    from dotenv import load_dotenv

    load_dotenv(ROOT / ".env")
    if not os.getenv("GPTUNNEL_API_KEY"):
        raise RuntimeError("Нет GPTUNNEL_API_KEY в .env")

    from model_benchmark.runner import run_lesson_benchmark

    live_report = run_lesson_benchmark(lesson_ids=["lesson-variables"])  # дым по умолчанию
    print("saved:", live_report.get("saved_to"))
    print_table(summarize_leaderboard(live_report))
else:
    print("Живой прогон отключён (RUN_LIVE=False). Используются сохранённые результаты.")